<a href="https://colab.research.google.com/github/durgesh-js/Helix/blob/main/Helix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous AI Development Manager

**Objective:** given a high-level software-development goal and a project (built-in demo or an uploaded ZIP), autonomously understand the project, plan work as a dependency-aware task graph, modify code through controlled tools, execute and test it, diagnose failures from real output, repair them, re-test, re-plan when needed, and produce a verified project archive and engineering report.

Closed loop: `GOAL -> UNDERSTAND -> PLAN -> SELECT MODEL -> ACT -> EXECUTE -> OBSERVE -> DIAGNOSE -> REPAIR -> RETEST -> REPLAN -> VERIFY`

This is not a "prompt in, code out" generator. Every action is driven by the actual state of the project (scan results, test output, error text). Run cells top to bottom in Google Colab.


## Setup

Dependencies, configuration, secure API key entry, and the model pool + resource monitor + router that every agent role below shares.


In [1]:
# Install dependencies. Minimal footprint: no agent frameworks, custom orchestration only.
%pip install -q -U google-genai pytest


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [2]:
import os, sys, io, re, json, time, uuid, shutil, zipfile, subprocess, tempfile, textwrap, traceback, platform
from dataclasses import dataclass, field, asdict
from enum import Enum
from pathlib import Path
from typing import Any, Optional

# ---- Global run configuration -------------------------------------------------
CONFIG = {
    "MAX_TOTAL_MODEL_CALLS": 80,
    "MAX_REPAIR_ATTEMPTS": 3,        # per task
    "MAX_TASK_ITERATIONS": 25,       # total tasks processed (incl. repair-spawned tasks)
    "MAX_SESSION_TIME_SECONDS": 40 * 60,
    "OPTIONAL_TOKEN_BUDGET": None,   # e.g. 500_000, or None to disable
    "AUTO_APPROVE_SAFE_ACTIONS": True,          # safe = write/create/read/list/search/run_tests
    "AUTO_APPROVE_DESTRUCTIVE_DEMO": True,      # only applies to the built-in demo project
    "AUTO_APPROVE_DESTRUCTIVE_UPLOAD": False,   # uploaded projects require explicit approval
    "TEST_TIMEOUT_SECONDS": 60,
    "RUN_TIMEOUT_SECONDS": 30,
}

SESSION_START = time.time()

def elapsed():
    return time.time() - SESSION_START

def session_time_left():
    return CONFIG["MAX_SESSION_TIME_SECONDS"] - elapsed()

print("Config loaded. Session time budget:", CONFIG["MAX_SESSION_TIME_SECONDS"], "seconds")

# ---- Execution console --------------------------------------------------------
# Defined early because most components log through it; EXECUTION_LOG backs the
# execution_log.json artifact produced at the end of the run.
EXECUTION_LOG = []

def console_log(tag: str, message: str):
    entry = {"tag": tag, "message": message, "t": round(elapsed(), 2)}
    EXECUTION_LOG.append(entry)
    print(f"[{tag}] {message}")



Config loaded. Session time budget: 2400 seconds


In [3]:
import getpass
from google import genai

# Never hard-code the key and never print it back.
_existing = os.environ.get("GEMINI_API_KEY")
if not _existing:
    _key = getpass.getpass("Enter your GEMINI_API_KEY (input hidden): ").strip()
    if not _key:
        raise RuntimeError("A Gemini API key is required to continue.")
    os.environ["GEMINI_API_KEY"] = _key

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Gemini client configured.")


Enter your GEMINI_API_KEY (input hidden): ··········
Gemini client configured.


In [4]:
# Model pool.
#
# NOTE ON MODEL NAMES: this notebook only uses Gemini model IDs documented as
# live/stable on ai.google.dev at the time this notebook was written. Model
# availability changes frequently -- if a model below has been retired or
# renamed by the time you run this, update MODEL_POOL only; the router,
# monitor, and every agent read model names from this table alone.

class Tier(str, Enum):
    COMPLEX = "complex"      # architecture, debugging, multi-file repair
    NORMAL = "normal"        # ordinary coding tasks
    LIGHT = "light"          # simple analysis / classification
    LOCAL = "local"          # offline fallback, no API dependency

MODEL_POOL = [
    {"name": "gemini-3-flash-preview", "tier": Tier.COMPLEX, "provider": "gemini", "rank": 0},
    {"name": "gemini-2.5-flash",       "tier": Tier.NORMAL,  "provider": "gemini", "rank": 1},
    {"name": "gemini-3.1-flash-lite",  "tier": Tier.LIGHT,   "provider": "gemini", "rank": 2},
    {"name": "qwen3-4b-local",         "tier": Tier.LOCAL,   "provider": "local",  "rank": 3},
]

TIER_ORDER = [Tier.COMPLEX, Tier.NORMAL, Tier.LIGHT, Tier.LOCAL]

def _models_at_or_below(tier: Tier):
    """Models whose tier is at the requested strength or weaker, ranked strongest-first."""
    idx = TIER_ORDER.index(tier)
    allowed = set(TIER_ORDER[idx:])
    return sorted([m for m in MODEL_POOL if m["tier"] in allowed], key=lambda m: m["rank"])


class ErrorClass(str, Enum):
    RATE_LIMIT = "rate_limit"
    QUOTA_EXHAUSTED = "quota_exhausted"
    AUTH_FAILURE = "auth_failure"
    MODEL_UNAVAILABLE = "model_unavailable"
    CONTEXT_LIMIT = "context_limit"
    NETWORK = "network"
    INVALID_RESPONSE = "invalid_response"
    TRANSIENT = "transient"
    UNKNOWN = "unknown"

def classify_api_error(exc: Exception) -> ErrorClass:
    """Classify a Gemini API exception from its message/status. We never assume
    remaining-credit information the API does not expose; we only classify the
    observed response/exception."""
    msg = str(exc).lower()
    status = getattr(exc, "status_code", None) or getattr(exc, "code", None)
    if status == 401 or "api key not valid" in msg or "unauthenticated" in msg or "permission" in msg:
        return ErrorClass.AUTH_FAILURE
    if status == 429 or "rate limit" in msg or "resource_exhausted" in msg:
        if "quota" in msg or "billing" in msg:
            return ErrorClass.QUOTA_EXHAUSTED
        return ErrorClass.RATE_LIMIT
    if status == 404 or "not found" in msg or "not supported" in msg:
        return ErrorClass.MODEL_UNAVAILABLE
    if "context" in msg and ("token" in msg or "too long" in msg or "exceeds" in msg):
        return ErrorClass.CONTEXT_LIMIT
    if isinstance(exc, (ConnectionError, TimeoutError)) or "timed out" in msg or "network" in msg:
        return ErrorClass.NETWORK
    if status in (500, 503) or "internal" in msg or "unavailable" in msg:
        return ErrorClass.TRANSIENT
    if "invalid" in msg and "response" in msg:
        return ErrorClass.INVALID_RESPONSE
    return ErrorClass.UNKNOWN


class ResourceMonitor:
    """Tracks only what we can actually observe. Token counts come from
    response.usage_metadata when the SDK provides it; otherwise we report
    'Unavailable' rather than fabricate a number."""

    def __init__(self):
        self.calls_by_model = {}
        self.successes_by_model = {}
        self.failures_by_model = {}
        self.fallback_calls = 0
        self.model_switches = []          # list of (from, to)
        self.input_tokens = 0
        self.output_tokens = 0
        self.token_metadata_seen = False
        self.task_count = 0
        self.repair_attempts = 0
        self.errors = []                  # list of dicts: model, error_class, message, ts
        self.current_model = None

    def total_calls(self):
        return sum(self.calls_by_model.values())

    def record_call(self, model_name, success, input_tokens=None, output_tokens=None, is_fallback=False):
        self.calls_by_model[model_name] = self.calls_by_model.get(model_name, 0) + 1
        bucket = self.successes_by_model if success else self.failures_by_model
        bucket[model_name] = bucket.get(model_name, 0) + 1
        if is_fallback:
            self.fallback_calls += 1
        if input_tokens is not None:
            self.input_tokens += input_tokens
            self.token_metadata_seen = True
        if output_tokens is not None:
            self.output_tokens += output_tokens
            self.token_metadata_seen = True
        if self.current_model and self.current_model != model_name:
            self.model_switches.append((self.current_model, model_name))
        self.current_model = model_name

    def record_error(self, model_name, error_class: "ErrorClass", message):
        self.errors.append({"model": model_name, "error_class": error_class.value,
                             "message": message[:300], "ts": time.time()})

    def limits_status(self):
        reasons = []
        if self.total_calls() >= CONFIG["MAX_TOTAL_MODEL_CALLS"]:
            reasons.append("MAX_TOTAL_MODEL_CALLS reached")
        if self.repair_attempts >= CONFIG["MAX_REPAIR_ATTEMPTS"] * CONFIG["MAX_TASK_ITERATIONS"]:
            reasons.append("global repair-attempt budget reached")
        if elapsed() >= CONFIG["MAX_SESSION_TIME_SECONDS"]:
            reasons.append("MAX_SESSION_TIME reached")
        budget = CONFIG.get("OPTIONAL_TOKEN_BUDGET")
        if budget and self.token_metadata_seen and (self.input_tokens + self.output_tokens) >= budget:
            reasons.append("OPTIONAL_TOKEN_BUDGET reached")
        return reasons

    def summary(self):
        return {
            "total_calls": self.total_calls(),
            "calls_by_model": dict(self.calls_by_model),
            "successes_by_model": dict(self.successes_by_model),
            "failures_by_model": dict(self.failures_by_model),
            "fallback_calls": self.fallback_calls,
            "model_switches": [f"{a} -> {b}" for a, b in self.model_switches],
            "input_tokens": self.input_tokens if self.token_metadata_seen else "Unavailable",
            "output_tokens": self.output_tokens if self.token_metadata_seen else "Unavailable",
            "task_count": self.task_count,
            "repair_attempts": self.repair_attempts,
            "session_seconds": round(elapsed(), 1),
            "current_model": self.current_model,
        }

MONITOR = ResourceMonitor()
print("Model pool:", [m["name"] for m in MODEL_POOL])


Model pool: ['gemini-3-flash-preview', 'gemini-2.5-flash', 'gemini-3.1-flash-lite', 'qwen3-4b-local']


In [5]:
from google.genai import types as gtypes

class ResourceLimitReached(Exception):
    pass

_QWEN_STATE = {"checked": False, "feasible": False, "reason": "not checked", "llm": None}

def qwen_feasible():
    """Decide, without loading anything, whether local Qwen3-4B (GGUF, Q4_K_M)
    execution is practical here. We never auto-download/load at startup."""
    if _QWEN_STATE["checked"]:
        return _QWEN_STATE["feasible"], _QWEN_STATE["reason"]
    _QWEN_STATE["checked"] = True
    try:
        import importlib
        has_llama_cpp = importlib.util.find_spec("llama_cpp") is not None
    except Exception:
        has_llama_cpp = False
    free_gb = None
    try:
        free_gb = shutil.disk_usage("/").free / (1024 ** 3)
    except Exception:
        pass
    has_gpu = False
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
        has_gpu = (r.returncode == 0)
    except Exception:
        has_gpu = False
    if not has_llama_cpp:
        _QWEN_STATE["reason"] = "llama-cpp-python not installed"
    elif free_gb is not None and free_gb < 6:
        _QWEN_STATE["reason"] = f"insufficient disk space ({free_gb:.1f} GB free, need ~6 GB for a Q4_K_M GGUF)"
    else:
        _QWEN_STATE["feasible"] = True
        _QWEN_STATE["reason"] = "GPU available" if has_gpu else "CPU-only execution possible but slow"
    return _QWEN_STATE["feasible"], _QWEN_STATE["reason"]


def local_qwen_generate(prompt: str):
    """Lazily load and run Qwen3-4B-GGUF via llama.cpp. Only called when the
    router has exhausted every Gemini option and qwen_feasible() is True."""
    feasible, reason = qwen_feasible()
    if not feasible:
        return None
    if _QWEN_STATE["llm"] is None:
        try:
            from llama_cpp import Llama
            model_path = os.environ.get("QWEN_GGUF_PATH")
            if not model_path or not os.path.exists(model_path):
                print("[ROUTER] Qwen3-4B GGUF path not set/found (QWEN_GGUF_PATH env var); "
                      "skipping local fallback.")
                return None
            _QWEN_STATE["llm"] = Llama(model_path=model_path, n_ctx=4096, verbose=False)
        except Exception as e:
            print(f"[ROUTER] Could not load local Qwen model: {e}")
            return None
    try:
        out = _QWEN_STATE["llm"](prompt, max_tokens=1024, stop=["</s>"])
        return out["choices"][0]["text"]
    except Exception as e:
        print(f"[ROUTER] Local Qwen inference failed: {e}")
        return None


class ModelRouter:
    """Picks the strongest AVAILABLE and SUITABLE model for a task, tracking
    per-model failures within the session so a model that just failed is not
    retried blindly, and never falling back to a weaker model when a stronger
    one is still available."""

    def __init__(self, monitor: ResourceMonitor):
        self.monitor = monitor
        self.unavailable_until = {}     # model_name -> epoch time it may be retried
        self.permanently_dead = set()   # e.g. auth failure applies to all Gemini models

    def _is_temporarily_down(self, model_name):
        until = self.unavailable_until.get(model_name)
        return until is not None and time.time() < until

    def _candidates(self, tier: Tier):
        cands = _models_at_or_below(tier)
        out = []
        for m in cands:
            if m["name"] in self.permanently_dead:
                continue
            if m["provider"] == "gemini" and self._is_temporarily_down(m["name"]):
                continue
            if m["provider"] == "local":
                feasible, _ = qwen_feasible()
                if not feasible:
                    continue
            out.append(m)
        return out

    def select(self, tier: Tier, previous_failures=None):
        previous_failures = previous_failures or []
        cands = [m for m in self._candidates(tier) if m["name"] not in previous_failures]
        if not cands:
            # every candidate already failed this call-chain; relax rather than dead-end
            cands = self._candidates(tier)
        if not cands:
            return None, "no suitable model currently available (all candidates down or infeasible)"
        chosen = cands[0]
        reason = f"top-ranked available model for tier={tier.value}"
        if previous_failures:
            reason = f"fallback after failures on {previous_failures}"
        return chosen, reason

    def penalize(self, model_name, error_class: ErrorClass):
        if error_class == ErrorClass.AUTH_FAILURE:
            for m in MODEL_POOL:            # a bad key kills every Gemini model
                if m["provider"] == "gemini":
                    self.permanently_dead.add(m["name"])
        elif error_class == ErrorClass.QUOTA_EXHAUSTED:
            self.unavailable_until[model_name] = time.time() + 600
        elif error_class == ErrorClass.RATE_LIMIT:
            self.unavailable_until[model_name] = time.time() + 30
        elif error_class == ErrorClass.MODEL_UNAVAILABLE:
            self.permanently_dead.add(model_name)
        # TRANSIENT / NETWORK / CONTEXT_LIMIT / INVALID_RESPONSE / UNKNOWN: no
        # standing penalty beyond "don't retry the same model this call-chain".

ROUTER = ModelRouter(MONITOR)


def route_and_call(task_complexity: Tier, system_instruction: str, contents,
                    response_json: bool = False, tools=None, max_model_attempts: int = 4):
    """Core entry point every agent role uses. Selects a model, calls it,
    classifies failures, penalizes/switches, and retries with the next best
    model -- never the same failed model, never an infinite loop. Returns
    (response_or_text, decision_log)."""
    tried = []
    decision_log = []
    for _ in range(max_model_attempts):
        if MONITOR.total_calls() >= CONFIG["MAX_TOTAL_MODEL_CALLS"]:
            raise ResourceLimitReached("MAX_TOTAL_MODEL_CALLS reached during routing")
        model, reason = ROUTER.select(task_complexity, previous_failures=tried)
        if model is None:
            raise RuntimeError(f"Model routing exhausted: {reason}")
        decision = {"selected_model": model["name"], "reason": reason,
                    "task_complexity": task_complexity.value, "fallback_trigger": None,
                    "previous_failures": list(tried)}

        if model["provider"] == "local":
            flat_prompt = contents if isinstance(contents, str) else str(contents)
            text = local_qwen_generate(f"{system_instruction}\n\n{flat_prompt}")
            if text is None:
                tried.append(model["name"])
                decision["fallback_trigger"] = "local_model_unavailable"
                decision_log.append(decision)
                continue
            MONITOR.record_call(model["name"], True, is_fallback=len(tried) > 0)
            decision_log.append(decision)
            return text, decision_log

        try:
            cfg_kwargs = {"system_instruction": system_instruction}
            if response_json:
                cfg_kwargs["response_mime_type"] = "application/json"
            if tools:
                cfg_kwargs["tools"] = tools
                # We dispatch function calls ourselves (see run_agent_with_tools);
                # explicitly disabling AFC avoids the SDK's "use Chat.send_message
                # instead" warning on every call and keeps behavior predictable.
                cfg_kwargs["automatic_function_calling"] = gtypes.AutomaticFunctionCallingConfig(disable=True)
            gen_cfg = gtypes.GenerateContentConfig(**cfg_kwargs)
            resp = client.models.generate_content(
                model=model["name"], contents=contents, config=gen_cfg)
            usage = getattr(resp, "usage_metadata", None)
            in_tok = getattr(usage, "prompt_token_count", None) if usage else None
            out_tok = getattr(usage, "candidates_token_count", None) if usage else None
            MONITOR.record_call(model["name"], True, in_tok, out_tok, is_fallback=len(tried) > 0)
            decision_log.append(decision)
            return resp, decision_log
        except Exception as e:
            err_class = classify_api_error(e)
            MONITOR.record_call(model["name"], False, is_fallback=len(tried) > 0)
            MONITOR.record_error(model["name"], err_class, str(e))
            ROUTER.penalize(model["name"], err_class)
            decision["fallback_trigger"] = err_class.value
            decision_log.append(decision)
            tried.append(model["name"])
            console_log("ROUTER", f"{model['name']} failed ({err_class.value}); selecting next model")
            continue
    raise RuntimeError(f"All model attempts exhausted for tier {task_complexity.value}: tried {tried}")


## Workspace and project

An isolated `/content/autodev_workspace/`, a small built-in Expense Tracker demo with real (undisclosed) defects, and safe ZIP upload/extraction for your own project.


In [6]:
WORKSPACE_ROOT = Path("/content/autodev_workspace") if Path("/content").exists() \
    else Path.cwd() / "autodev_workspace"
if WORKSPACE_ROOT.exists():
    shutil.rmtree(WORKSPACE_ROOT)
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = WORKSPACE_ROOT / "project"
PROJECT_ROOT.mkdir(exist_ok=True)
IS_DEMO_PROJECT = True   # flips to False once/if a real ZIP is loaded

print("Workspace:", WORKSPACE_ROOT)


def build_demo_project(root: Path):
    """A small Expense Tracker with realistic, non-obvious defects for the
    agent to discover through analysis and testing -- it is not told where
    the bugs are."""
    (root / "tests").mkdir(exist_ok=True)

    (root / "README.md").write_text(
        "# Expense Tracker\n\nTrack personal expenses by category with basic "
        "reporting.\n\nRun `python app.py` for a CLI demo, `pytest` for tests.\n"
    )
    (root / "requirements.txt").write_text("pytest>=7.0\n")

    validators_src = (
        '"""Input validation helpers for the expense tracker."""\n\n'
        'ALLOWED_CATEGORIES = {"food", "travel", "utilities", "entertainment", "other"}\n\n'
        "def validate_amount(amount):\n"
        "    # Defect: accepts negative and zero amounts, and never checks type.\n"
        "    if amount is None:\n"
        '        raise ValueError("amount is required")\n'
        "    return float(amount)\n\n"
        "def validate_category(category):\n"
        "    if category not in ALLOWED_CATEGORIES:\n"
        '        raise ValueError(f"unknown category: {category}")\n'
        "    return category\n\n"
        "def validate_description(description):\n"
        "    if not description or not description.strip():\n"
        '        raise ValueError("description is required")\n'
        "    return description.strip()\n"
    )
    (root / "validators.py").write_text(validators_src)

    database_src = (
        '"""A tiny in-memory store standing in for a real database."""\n\n'
        "class InMemoryStore:\n"
        "    def __init__(self):\n"
        "        self._rows = []\n"
        "        self._next_id = 1\n\n"
        "    def insert(self, record: dict) -> int:\n"
        "        record = dict(record)\n"
        '        record["id"] = self._next_id\n'
        "        self._rows.append(record)\n"
        "        self._next_id += 1\n"
        '        return record["id"]\n\n'
        "    def all(self):\n"
        "        return list(self._rows)\n\n"
        "    def by_category(self, category):\n"
        '        return [r for r in self._rows if r["category"] == category]\n'
    )
    (root / "database.py").write_text(database_src)

    expenses_src = (
        '"""Core expense-tracking logic."""\n'
        "from database import InMemoryStore\n"
        "from validators import validate_amount, validate_category, validate_description\n\n"
        "class ExpenseTracker:\n"
        "    def __init__(self, store=None):\n"
        "        self.store = store or InMemoryStore()\n\n"
        "    def add_expense(self, amount, category, description):\n"
        "        amount = validate_amount(amount)\n"
        "        category = validate_category(category)\n"
        "        description = validate_description(description)\n"
        "        return self.store.insert({\n"
        '            "amount": amount, "category": category, "description": description,\n'
        "        })\n\n"
        "    def total(self):\n"
        '        return sum(r["amount"] for r in self.store.all())\n\n'
        "    def total_by_category(self, category):\n"
        "        # Defect: rounds using int() which truncates toward zero instead\n"
        "        # of preserving the precise decimal total.\n"
        "        rows = self.store.by_category(category)\n"
        '        raw_total = sum(r["amount"] for r in rows)\n'
        "        return int(raw_total)\n\n"
        "    def average(self):\n"
        "        rows = self.store.all()\n"
        "        if not rows:\n"
        "            return 0\n"
        "        # Defect: integer division drops fractional cents.\n"
        "        return self.total() // len(rows)\n"
    )
    (root / "expenses.py").write_text(expenses_src)

    app_src = (
        '"""CLI entry point."""\n'
        "from expenses import ExpenseTracker\n\n"
        "def main():\n"
        "    tracker = ExpenseTracker()\n"
        '    tracker.add_expense(12.50, "food", "lunch")\n'
        '    tracker.add_expense(40.00, "travel", "taxi")\n'
        '    print("Total:", tracker.total())\n\n'
        'if __name__ == "__main__":\n'
        "    main()\n"
    )
    (root / "app.py").write_text(app_src)

    (root / "tests" / "__init__.py").write_text("")
    tests_src = (
        "from expenses import ExpenseTracker\n\n"
        "def test_add_and_total():\n"
        "    t = ExpenseTracker()\n"
        '    t.add_expense(10, "food", "snack")\n'
        '    t.add_expense(5, "food", "coffee")\n'
        "    assert t.total() == 15\n\n"
        "def test_average_is_accurate():\n"
        "    t = ExpenseTracker()\n"
        '    t.add_expense(10, "food", "a")\n'
        '    t.add_expense(3, "food", "b")\n'
        "    # 13 / 2 = 6.5, not 6 -- exposes the integer-division defect.\n"
        "    assert t.average() == 6.5\n\n"
        "def test_total_by_category_matches_precise_sum():\n"
        "    t = ExpenseTracker()\n"
        '    t.add_expense(1.20, "travel", "bus")\n'
        '    t.add_expense(1.20, "travel", "bus")\n'
        "    # 2.40 total -- exposes the int() truncation defect.\n"
        '    assert t.total_by_category("travel") == 2.40\n'
    )
    (root / "tests" / "test_expenses.py").write_text(tests_src)
    print("Demo project written:", sorted(p.name for p in root.iterdir()))


def safe_extract_zip(zip_path: Path, dest: Path):
    """Extract a ZIP into dest while blocking path traversal, absolute paths,
    and symlink escapes."""
    dest = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            name = member.filename
            if name.startswith("/") or ":" in name or ".." in Path(name).parts:
                raise ValueError(f"Unsafe path in archive, refusing to extract: {name}")
            target = (dest / name).resolve()
            if not str(target).startswith(str(dest)):
                raise ValueError(f"Path traversal detected, refusing to extract: {name}")
        zf.extractall(dest)
    for p in dest.rglob("*"):     # reject any symlink that escaped the workspace
        if p.is_symlink():
            resolved = p.resolve()
            if not str(resolved).startswith(str(dest)):
                p.unlink()
                print(f"Removed unsafe symlink: {p}")


def load_uploaded_project():
    """Colab-only: prompts a ZIP upload and safely extracts it. Falls back to
    the demo project if not running in Colab or nothing is uploaded."""
    global IS_DEMO_PROJECT
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab (or upload widget unavailable) -- using demo project.")
        build_demo_project(PROJECT_ROOT)
        return
    print("Upload a project ZIP, or interrupt/skip this cell to use the built-in demo.")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded -- using demo project.")
        build_demo_project(PROJECT_ROOT)
        return
    fname = next(iter(uploaded))
    tmp_zip = WORKSPACE_ROOT / fname
    tmp_zip.write_bytes(uploaded[fname])
    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
    PROJECT_ROOT.mkdir(exist_ok=True)
    safe_extract_zip(tmp_zip, PROJECT_ROOT)
    IS_DEMO_PROJECT = False
    print("Project extracted to", PROJECT_ROOT)


# --- Choose ONE of the two lines below -----------------------------------------
build_demo_project(PROJECT_ROOT)          # built-in demo (default for this run)
# load_uploaded_project()                 # uncomment to upload your own ZIP instead


Workspace: /content/autodev_workspace
Demo project written: ['README.md', 'app.py', 'database.py', 'expenses.py', 'requirements.txt', 'tests', 'validators.py']


## Project understanding and controlled tools

A project scanner, safe file tools (path-validated, change-tracked), a token-efficient context builder, and subprocess-based execution/testing tools.


In [7]:
SOURCE_EXTS = {".py"}
TEST_HINTS = ("test_", "_test.py", "tests/")

def scan_project(root: Path) -> dict:
    """Builds a compact manifest: files, source files, tests, config/metadata
    files, entry points, and a best-effort language/framework guess. The MVP
    is strongest for Python; other languages are reported with reduced
    (read-only) execution capability rather than claimed as fully supported."""
    all_files, source_files, test_files, config_files = [], [], [], []
    entry_points = []
    lang_counts = {}

    for p in root.rglob("*"):
        if p.is_dir():
            continue
        rel = str(p.relative_to(root))
        if "/.git/" in f"/{rel}" or rel.startswith(".git"):
            continue
        all_files.append(rel)
        ext = p.suffix.lower()
        lang_counts[ext] = lang_counts.get(ext, 0) + 1
        if ext in SOURCE_EXTS:
            is_test = ("test" in p.name.lower()) or ("tests" in p.parts)
            (test_files if is_test else source_files).append(rel)
            if not is_test and ext == ".py":
                try:
                    text = p.read_text(errors="ignore")
                    if "__main__" in text:
                        entry_points.append(rel)
                except Exception:
                    pass
        if p.name in ("requirements.txt", "pyproject.toml", "setup.py", "package.json",
                      "Pipfile", "pytest.ini", "setup.cfg"):
            config_files.append(rel)

    primary_lang = ".py"
    if lang_counts:
        primary_lang = max(lang_counts, key=lambda k: lang_counts[k] if k else 0)
    is_python_project = ".py" in lang_counts

    manifest = {
        "root": str(root),
        "total_files": len(all_files),
        "source_files": sorted(source_files),
        "test_files": sorted(test_files),
        "config_files": sorted(config_files),
        "entry_points": sorted(entry_points),
        "has_readme": any(f.lower().startswith("readme") for f in all_files),
        "primary_extension": primary_lang,
        "is_python_project": is_python_project,
        "execution_supported": is_python_project,
    }
    if not is_python_project:
        console_log("MANAGER", "Non-Python project detected: execution/testing tools are "
                                "read-only (scan/search) only, not run/repair.")
    return manifest

PROJECT_MANIFEST = scan_project(PROJECT_ROOT)
console_log("MANAGER", f"Project scanned: {PROJECT_MANIFEST['total_files']} files, "
                        f"{len(PROJECT_MANIFEST['source_files'])} source, "
                        f"{len(PROJECT_MANIFEST['test_files'])} test files")


[MANAGER] Project scanned: 8 files, 4 source, 2 test files


In [8]:
class FileChangeLog:
    def __init__(self):
        self.created_files = []
        self.modified_files = []
        self.deleted_files = []

FILE_LOG = FileChangeLog()

def _safe_resolve(rel_path: str) -> Path:
    """Resolve rel_path against PROJECT_ROOT, refusing traversal, absolute
    paths, and any resolution that escapes the workspace."""
    if rel_path is None:
        raise ValueError("path is required")
    if os.path.isabs(rel_path) or ".." in Path(rel_path).parts:
        raise PermissionError(f"unsafe path rejected: {rel_path}")
    target = (PROJECT_ROOT / rel_path).resolve()
    root = PROJECT_ROOT.resolve()
    if not str(target).startswith(str(root)):
        raise PermissionError(f"path escapes workspace root: {rel_path}")
    return target


def list_files(subdir: str = ".") -> list:
    base = _safe_resolve(subdir)
    if not base.exists():
        return []
    return sorted(str(p.relative_to(PROJECT_ROOT)) for p in base.rglob("*") if p.is_file())


def read_file(path: str, max_chars: int = 8000) -> str:
    target = _safe_resolve(path)
    if not target.exists():
        raise FileNotFoundError(path)
    text = target.read_text(errors="ignore")
    if len(text) > max_chars:
        return text[:max_chars] + f"\n... [truncated, {len(text) - max_chars} more chars]"
    return text


def search_code(query: str, subdir: str = ".") -> list:
    """Substring/regex search across source files; returns file:line matches
    so the model can request only the relevant file next, instead of the
    whole repository."""
    base = _safe_resolve(subdir)
    try:
        pattern = re.compile(query)
    except re.error:
        pattern = re.compile(re.escape(query))
    hits = []
    for p in base.rglob("*.py"):
        try:
            for i, line in enumerate(p.read_text(errors="ignore").splitlines(), start=1):
                if pattern.search(line):
                    hits.append(f"{p.relative_to(PROJECT_ROOT)}:{i}: {line.strip()[:200]}")
        except Exception:
            continue
    return hits[:100]


def write_file(path: str, content: str) -> str:
    target = _safe_resolve(path)
    existed = target.exists()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    rel = str(target.relative_to(PROJECT_ROOT))
    (FILE_LOG.modified_files if existed else FILE_LOG.created_files).append(rel)
    console_log("CODER", f"{'Modified' if existed else 'Created'} {rel}")
    return f"wrote {len(content)} chars to {rel}"


def create_file(path: str, content: str = "") -> str:
    target = _safe_resolve(path)
    if target.exists():
        raise FileExistsError(f"{path} already exists, use write_file to modify it")
    return write_file(path, content)


def file_exists(path: str) -> bool:
    return _safe_resolve(path).exists()


def get_file_metadata(path: str) -> dict:
    target = _safe_resolve(path)
    if not target.exists():
        return {"exists": False}
    stat = target.stat()
    return {"exists": True, "size_bytes": stat.st_size, "modified": stat.st_mtime,
            "lines": target.read_text(errors="ignore").count("\n") + 1}


def delete_file(path: str, approved: bool = False) -> str:
    target = _safe_resolve(path)
    if not target.exists():
        return f"{path} does not exist, nothing to delete"
    if not approve_action("delete_file", {"path": path}, pre_approved=approved):
        return f"deletion of {path} requires approval and was not approved; skipped"
    target.unlink()
    rel = str(target.relative_to(PROJECT_ROOT))
    FILE_LOG.deleted_files.append(rel)
    console_log("CODER", f"Deleted {rel} (approved)")
    return f"deleted {rel}"


In [9]:
def build_task_context(task: dict, project_state: dict, max_chars: int = 6000) -> str:
    """Token-efficient context: only what this task needs, not the whole repo.
    Callers should further narrow with search_code before reading full files;
    this just assembles what has already been gathered for the prompt."""
    parts = [
        f"GOAL: {project_state['project_goal']}",
        f"TASK: {task['title']}\n{task['description']}",
        f"ACCEPTANCE CRITERIA: {task.get('acceptance_criteria', 'n/a')}",
    ]
    for f in task.get("relevant_files", [])[:5]:
        try:
            parts.append(f"--- {f} ---\n{read_file(f, max_chars=1500)}")
        except Exception as e:
            parts.append(f"--- {f} (unreadable: {e}) ---")
    if task.get("previous_attempt_summary"):
        parts.append(f"PREVIOUS ATTEMPT: {task['previous_attempt_summary']}")
    if task.get("last_error"):
        parts.append(f"CURRENT ERROR OUTPUT:\n{task['last_error'][-2000:]}")
    context = "\n\n".join(parts)
    if len(context) > max_chars:
        context = context[:max_chars] + "\n... [context truncated for token efficiency]"
    return context


In [10]:
def run_python(args: list, timeout: int = None) -> dict:
    """Runs `python <args>` with the project as cwd, captured output, and a
    timeout. No shell=True, no arbitrary shell strings."""
    timeout = timeout or CONFIG["RUN_TIMEOUT_SECONDS"]
    cmd = [sys.executable] + list(args)
    try:
        result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True,
                                 text=True, timeout=timeout)
        return {"exit_code": result.returncode, "stdout": result.stdout[-4000:],
                "stderr": result.stderr[-4000:], "timed_out": False}
    except subprocess.TimeoutExpired as e:
        return {"exit_code": None, "stdout": (e.stdout or "")[-4000:],
                "stderr": (e.stderr or "") + "\n[TIMEOUT]", "timed_out": True}


def run_tests(target: str = "tests", timeout: int = None) -> dict:
    """Runs pytest (falls back to unittest discovery if pytest is missing) on
    the given target, focused when possible. Never claims tests passed
    without actually executing them."""
    if not PROJECT_MANIFEST.get("is_python_project", True):
        return {"ran": False, "reason": "non-Python project: execution not supported"}
    timeout = timeout or CONFIG["TEST_TIMEOUT_SECONDS"]
    t0 = time.time()
    target_path = PROJECT_ROOT / target
    if not target_path.exists():
        return {"ran": False, "reason": f"no tests found at {target}"}
    result = run_python(["-m", "pytest", target, "-q", "--no-header"], timeout=timeout)
    duration = round(time.time() - t0, 2)
    out = result["stdout"] + result["stderr"]
    m = re.search(r"(\d+) failed", out)
    failed = int(m.group(1)) if m else (0 if result["exit_code"] == 0 else None)
    m = re.search(r"(\d+) passed", out)
    passed = int(m.group(1)) if m else None
    return {
        "ran": True, "exit_code": result["exit_code"], "passed": passed, "failed": failed,
        "duration_seconds": duration, "stdout": result["stdout"], "stderr": result["stderr"],
        "success": result["exit_code"] == 0,
    }


def syntax_check(paths: list) -> dict:
    """Compiles each file with py_compile to catch syntax errors before running
    the full test suite (cheap, fast signal)."""
    import py_compile
    errors = {}
    for rel in paths:
        try:
            py_compile.compile(str(PROJECT_ROOT / rel), doraise=True)
        except Exception as e:
            errors[rel] = str(e)
    return {"ok": not errors, "errors": errors}


## State, task graph, and safety

Persistent project state, the dependency-aware task graph, and the approval gate for destructive actions.


In [11]:
class TaskStatus(str, Enum):
    PENDING = "PENDING"
    READY = "READY"
    RUNNING = "RUNNING"
    BLOCKED = "BLOCKED"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"
    SKIPPED = "SKIPPED"


class ProjectState:
    def __init__(self, project_goal: str, deadline: str = "3-5 days (planning constraint only)"):
        self.project_goal = project_goal
        self.project_root = str(PROJECT_ROOT)
        self.project_manifest = PROJECT_MANIFEST
        self.tasks = []               # list of task dicts (the task graph)
        self.completed_tasks = []
        self.failed_tasks = []
        self.repair_attempts = {}     # task_id -> count
        self.test_results = []
        self.errors = []
        self.model_usage = {}
        self.execution_history = []
        self.status = "IN_PROGRESS"
        self.deadline = deadline
        self.start_time = time.time()
        self.last_update = time.time()

    def as_dict(self):
        d = dict(self.__dict__)
        d["changed_files"] = {
            "created": FILE_LOG.created_files,
            "modified": FILE_LOG.modified_files,
            "deleted": FILE_LOG.deleted_files,
        }
        return d

    def save(self, path: Path):
        self.last_update = time.time()
        path.write_text(json.dumps(self.as_dict(), indent=2, default=str))

    def __getitem__(self, key):     # dict-style access used by build_task_context()
        return getattr(self, key)


class TaskGraph:
    def __init__(self, state: ProjectState):
        self.state = state

    def add(self, title, description, priority=2, dependencies=None,
            estimated_complexity=Tier.NORMAL, relevant_files=None, acceptance_criteria=""):
        task = {
            "id": str(uuid.uuid4())[:8], "title": title, "description": description,
            "priority": priority, "status": TaskStatus.PENDING.value,
            "dependencies": dependencies or [], "estimated_complexity": estimated_complexity.value,
            "relevant_files": relevant_files or [], "acceptance_criteria": acceptance_criteria,
            "attempt_count": 0, "result": None, "last_error": None,
            "previous_attempt_summary": None,
        }
        self.state.tasks.append(task)
        self._refresh_ready()
        return task

    def _refresh_ready(self):
        done_ids = {t["id"] for t in self.state.tasks if t["status"] == TaskStatus.COMPLETED.value}
        for t in self.state.tasks:
            if t["status"] == TaskStatus.PENDING.value:
                if all(dep in done_ids for dep in t["dependencies"]):
                    t["status"] = TaskStatus.READY.value

    def next_ready_task(self):
        self._refresh_ready()
        ready = [t for t in self.state.tasks if t["status"] == TaskStatus.READY.value]
        if not ready:
            return None
        ready.sort(key=lambda t: t["priority"])
        return ready[0]

    def set_status(self, task_id, status: TaskStatus, result=None, error=None):
        for t in self.state.tasks:
            if t["id"] == task_id:
                t["status"] = status.value
                if result is not None:
                    t["result"] = result
                if error is not None:
                    t["last_error"] = error
                break
        self._refresh_ready()

    def all_terminal(self):
        active = {TaskStatus.PENDING.value, TaskStatus.READY.value,
                  TaskStatus.RUNNING.value, TaskStatus.BLOCKED.value}
        return not any(t["status"] in active for t in self.state.tasks)

    def add_repair_task(self, parent_task, diagnosis: dict):
        return self.add(
            title=f"Repair: {parent_task['title']}",
            description=f"Fix root cause identified during diagnosis: {diagnosis.get('root_cause')}",
            priority=0, dependencies=[],
            estimated_complexity=Tier.COMPLEX,
            relevant_files=diagnosis.get("relevant_files", parent_task["relevant_files"]),
            acceptance_criteria=parent_task["acceptance_criteria"],
        )


In [12]:
SAFE_ACTIONS = {"list_files", "read_file", "search_code", "write_file", "create_file",
                 "file_exists", "get_file_metadata", "run_python", "run_tests",
                 "get_project_state", "create_checkpoint"}
DESTRUCTIVE_ACTIONS = {"delete_file", "install_dependency", "destructive_command"}

APPROVAL_CALLBACK = None   # set to a function(action, details) -> bool for interactive use

def approve_action(action: str, details: dict, pre_approved: bool = False) -> bool:
    """Central gate for every tool call the model requests. Safe actions are
    auto-approved by policy; destructive actions require approval, and the
    default policy is stricter for uploaded projects than for the demo."""
    if pre_approved:
        return True
    if action in SAFE_ACTIONS and CONFIG["AUTO_APPROVE_SAFE_ACTIONS"]:
        return True
    if action in DESTRUCTIVE_ACTIONS:
        if APPROVAL_CALLBACK is not None:
            return bool(APPROVAL_CALLBACK(action, details))
        auto_ok = (CONFIG["AUTO_APPROVE_DESTRUCTIVE_DEMO"] if IS_DEMO_PROJECT
                   else CONFIG["AUTO_APPROVE_DESTRUCTIVE_UPLOAD"])
        console_log("SAFETY", f"{'Auto-approved' if auto_ok else 'Blocked'} destructive "
                               f"action '{action}' on {details} "
                               f"({'demo' if IS_DEMO_PROJECT else 'uploaded'} project policy)")
        return auto_ok
    return False   # unknown action type: deny by default


BLOCKED_COMMAND_PATTERNS = [
    r"rm\s+-rf\s+/", r"sudo\b", r"\bssh\b", r"\bscp\b", r"credential", r"secret",
    r"\.aws/", r"\.ssh/", r"\benv\b.*key", r"shutdown", r"reboot",
]

def is_command_blocked(command: str) -> bool:
    low = command.lower()
    return any(re.search(pat, low) for pat in BLOCKED_COMMAND_PATTERNS)


In [13]:
def get_project_state() -> dict:
    return STATE.as_dict()


def create_checkpoint(label: str = "") -> str:
    """Lightweight rollback point: snapshots current file contents under
    workspace/checkpoints/<n>_<label>/. No git/GitHub dependency."""
    ckpts_root = WORKSPACE_ROOT / "checkpoints"
    ckpts_root.mkdir(exist_ok=True)
    n = len(list(ckpts_root.glob("*")))
    ckpt_dir = ckpts_root / f"{n}_{label or 'ckpt'}"
    shutil.copytree(PROJECT_ROOT, ckpt_dir, dirs_exist_ok=True)
    console_log("SAFETY", f"Checkpoint created: {ckpt_dir.name}")
    return str(ckpt_dir)


# Tool dispatch table: every name here is an allowed, controlled operation.
# The model can only ever trigger these -- never an arbitrary shell command.
TOOL_DISPATCH = {
    "list_files": list_files,
    "read_file": read_file,
    "search_code": search_code,
    "write_file": write_file,
    "create_file": create_file,
    "delete_file": delete_file,
    "file_exists": file_exists,
    "get_file_metadata": get_file_metadata,
    "run_python": run_python,
    "run_tests": run_tests,
    "get_project_state": get_project_state,
    "create_checkpoint": create_checkpoint,
}

# JSON-schema declarations for each tool, used to build Gemini FunctionDeclarations.
TOOL_SCHEMAS = {
    "list_files": {"description": "List files under a project subdirectory.",
                   "params": {"subdir": ("string", "Subdirectory, default '.'")}},
    "read_file": {"description": "Read a project file's contents (truncated if large).",
                  "params": {"path": ("string", "Relative file path"),
                             "max_chars": ("integer", "Max characters to return")}},
    "search_code": {"description": "Search project source files by substring/regex.",
                     "params": {"query": ("string", "Search text or regex"),
                                "subdir": ("string", "Subdirectory to search, default '.'")}},
    "write_file": {"description": "Overwrite or create a file with new content.",
                   "params": {"path": ("string", "Relative file path"),
                              "content": ("string", "Full new file content")}},
    "create_file": {"description": "Create a new file (fails if it already exists).",
                    "params": {"path": ("string", "Relative file path"),
                               "content": ("string", "Initial file content")}},
    "delete_file": {"description": "Delete a file. Destructive: requires approval.",
                    "params": {"path": ("string", "Relative file path")}},
    "file_exists": {"description": "Check whether a file exists.",
                    "params": {"path": ("string", "Relative file path")}},
    "get_file_metadata": {"description": "Get size/line-count metadata for a file.",
                          "params": {"path": ("string", "Relative file path")}},
    "run_python": {"description": "Run `python <args>` inside the project, captured output.",
                   "params": {"args": ("array", "Argument list, e.g. ['app.py']")}},
    "run_tests": {"description": "Run the project's test suite (pytest).",
                  "params": {"target": ("string", "Test path, default 'tests'")}},
    "get_project_state": {"description": "Return the current persistent project state.",
                          "params": {}},
    "create_checkpoint": {"description": "Snapshot current files for rollback.",
                          "params": {"label": ("string", "Checkpoint label")}},
}

def build_tool_declarations(names):
    decls = []
    for name in names:
        spec = TOOL_SCHEMAS[name]
        props = {p: gtypes.Schema(type=t.upper()) for p, (t, _d) in spec["params"].items()}
        decls.append(gtypes.FunctionDeclaration(
            name=name, description=spec["description"],
            parameters=gtypes.Schema(type="OBJECT", properties=props) if props else None,
        ))
    return decls


def run_agent_with_tools(system_instruction: str, user_prompt: str, tool_names: list,
                          task_complexity: Tier, max_turns: int = 6) -> tuple:
    """Generic controlled function-calling loop shared by every agent role.
    The model may only invoke tools from TOOL_DISPATCH; every call is executed
    locally, gated by approve_action() where relevant, and the result is fed
    back before the next turn. Returns (final_text, transcript, decision_log)."""
    tools = [gtypes.Tool(function_declarations=build_tool_declarations(tool_names))]
    contents = [gtypes.Content(role="user", parts=[gtypes.Part(text=user_prompt)])]
    transcript = []
    all_decisions = []

    for turn in range(max_turns):
        resp, decisions = route_and_call(task_complexity, system_instruction, contents, tools=tools)
        all_decisions.extend(decisions)
        candidate = resp.candidates[0]
        parts = candidate.content.parts or []
        function_calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        text_parts = [p.text for p in parts if getattr(p, "text", None)]

        contents.append(candidate.content)   # keep the model's turn in history

        if not function_calls:
            return "\n".join(text_parts), transcript, all_decisions

        response_parts = []
        for fc in function_calls:
            name, args = fc.name, dict(fc.args or {})
            if name not in TOOL_DISPATCH:
                result = {"error": f"unknown tool '{name}'"}
            elif name in DESTRUCTIVE_ACTIONS or name == "delete_file":
                approved = approve_action(name, args)
                try:
                    result = TOOL_DISPATCH[name](**args, approved=approved) if name == "delete_file" \
                        else (TOOL_DISPATCH[name](**args) if approved else {"error": "not approved"})
                except Exception as e:
                    result = {"error": str(e)}
            else:
                try:
                    result = TOOL_DISPATCH[name](**args)
                except Exception as e:
                    result = {"error": f"{type(e).__name__}: {e}"}
            transcript.append({"tool": name, "args": args, "result": str(result)[:500]})
            response_parts.append(gtypes.Part.from_function_response(name=name, response={"result": result}))
        contents.append(gtypes.Content(role="tool", parts=response_parts))

    return "[max tool-calling turns reached without a final answer]", transcript, all_decisions


## Agent roles

Manager, Analyzer, Coding Agent, Testing Agent, Diagnosis/Repair Agent, and Replanner -- logical roles sharing the same routed model infrastructure and controlled tools, driven through Gemini function calling.


In [14]:
def _extract_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(json)?", "", text).rstrip("`").strip()
    return json.loads(text)


def analyzer_run(manifest: dict) -> dict:
    """ANALYZER role: examines the project with tools (search/read) and
    reports concrete problems and the files relevant to fixing them -- it
    does not modify anything."""
    system = ("You are the ANALYZER in an autonomous software engineering system. "
              "Use the available tools (list_files, read_file, search_code) to inspect "
              "the project. Do not guess: only report issues you actually observed in "
              "file contents. Budget yourself to at most 5-6 tool calls total, then STOP "
              "investigating and give your final answer even if you have not seen every "
              "file -- an incomplete-but-returned analysis is far more useful than none. "
              "When done, reply with ONLY a JSON object (no prose, no code fences) with "
              "fields: summary (string), issues (array of {description, files, severity}), "
              "key_files (array of strings).")
    prompt = f"Project manifest:\n{json.dumps(manifest, indent=2)}\n\nAnalyze this project."
    text, transcript, decisions = run_agent_with_tools(
        system, prompt, ["list_files", "read_file", "search_code"], Tier.NORMAL, max_turns=9)
    console_log("ANALYZER", f"Used {len(transcript)} tool calls; producing findings")
    try:
        return _extract_json(text)
    except Exception:
        return {"summary": text[:500], "issues": [], "key_files": manifest.get("source_files", [])}


def manager_create_plan(project_goal: str, manifest: dict, analysis: dict) -> list:
    """MANAGER role: turns the goal + analysis into a dependency-aware task
    list. Returns a list of task-spec dicts (not yet TaskGraph entries)."""
    system = ("You are the MANAGER in an autonomous software engineering system. "
              "Given a goal, a project manifest, and an analysis of concrete issues, "
              "produce a task graph. Prefer 3-7 small, independently testable tasks over "
              "one huge task. Reply with ONLY a JSON array (no prose, no code fences); each "
              "item: {title, description, priority (0=highest), dependencies (array of "
              "0-based indices into this array), estimated_complexity (one of 'complex', "
              "'normal', 'light'), relevant_files (array of paths), acceptance_criteria}.")
    prompt = (f"GOAL: {project_goal}\n\nMANIFEST:\n{json.dumps(manifest, indent=2)}\n\n"
              f"ANALYSIS:\n{json.dumps(analysis, indent=2)}")
    resp, decisions = route_and_call(Tier.COMPLEX, system, prompt, response_json=True)
    text = resp.text if hasattr(resp, "text") else str(resp)
    try:
        plan = json.loads(text)
    except Exception:
        plan = [{"title": "Improve project per objective",
                 "description": project_goal, "priority": 0, "dependencies": [],
                 "estimated_complexity": "normal",
                 "relevant_files": manifest.get("source_files", []),
                 "acceptance_criteria": "tests pass"}]
    console_log("PLANNER", f"{len(plan)} tasks created")
    return plan


def materialize_plan(graph: "TaskGraph", plan: list):
    """Converts index-based dependencies from the LLM plan into real task IDs."""
    tier_map = {"complex": Tier.COMPLEX, "normal": Tier.NORMAL, "light": Tier.LIGHT}
    created = []
    for item in plan:
        task = graph.add(
            title=item.get("title", "Untitled task"),
            description=item.get("description", ""),
            priority=item.get("priority", 2),
            dependencies=[],   # resolved in a second pass below
            estimated_complexity=tier_map.get(item.get("estimated_complexity", "normal"), Tier.NORMAL),
            relevant_files=item.get("relevant_files", []),
            acceptance_criteria=item.get("acceptance_criteria", ""),
        )
        created.append(task)
    for item, task in zip(plan, created):
        deps = item.get("dependencies", []) or []
        task["dependencies"] = [created[i]["id"] for i in deps if isinstance(i, int) and 0 <= i < len(created)]
    graph._refresh_ready()
    return created


In [15]:
def coding_agent_run(task: dict, state: ProjectState) -> dict:
    """CODING AGENT: implements the task using safe file tools only. Must
    actually call write_file/create_file to make changes -- text-only replies
    with no tool calls are treated as a non-attempt."""
    tier = {"complex": Tier.COMPLEX, "normal": Tier.NORMAL, "light": Tier.LIGHT}.get(
        task["estimated_complexity"], Tier.NORMAL)
    system = ("You are the CODING AGENT in an autonomous software engineering system. "
               "Implement the task by reading relevant files and calling write_file/"
               "create_file with complete, correct file content. Use search_code/read_file "
               "before editing so your changes match the existing code. Budget yourself to "
               "at most 6-7 tool calls, then stop and give your final answer. When finished, "
               "reply with ONLY a JSON object (no prose): {summary, files_changed}.")
    context = build_task_context(task, state.as_dict())
    text, transcript, decisions = run_agent_with_tools(
        system, context, ["list_files", "read_file", "search_code", "write_file", "create_file",
                           "file_exists", "get_file_metadata"], tier, max_turns=8)
    console_log("CODER", f"Task '{task['title']}': {len(transcript)} tool calls")
    try:
        summary = _extract_json(text)
    except Exception:
        summary = {"summary": text[:400], "files_changed": [c["args"].get("path") for c in transcript
                                                             if c["tool"] in ("write_file", "create_file")]}
    return summary


def testing_agent_run(scope: str = "tests") -> dict:
    """TESTING AGENT: actually executes tests and reports objective evidence.
    Never fabricates a pass/fail result."""
    result = run_tests(scope)
    STATE.test_results.append({"scope": scope, "t": elapsed(), **result})
    if not result.get("ran"):
        console_log("TESTER", f"Could not run tests: {result.get('reason')}")
    else:
        console_log("TESTER", f"{result.get('passed')} passed, {result.get('failed')} failed "
                               f"(exit={result.get('exit_code')})")
    return result


In [16]:
FAILURE_CATEGORIES = ["syntax", "import_dependency", "runtime", "logic", "test_expectation",
                      "configuration", "environment", "timeout", "model_api", "unknown"]

def diagnose_failure(task: dict, test_result: dict) -> dict:
    """DIAGNOSIS/REPAIR AGENT (diagnosis phase): classify the failure using the
    actual error output and identify a root cause + relevant files. No repair
    is applied here -- classification only."""
    system = ("You are the DIAGNOSIS AGENT. Classify this test failure using ONLY the "
              "actual output given (do not invent errors). Reply with ONLY a JSON object: "
              "{category (one of " + ", ".join(FAILURE_CATEGORIES) + "), root_cause, "
              "relevant_files (array), repair_plan (short string)}.")
    stdout = test_result.get("stdout", "")
    stderr = test_result.get("stderr", "")
    prompt = (f"Task: {task['title']}\nRelevant files: {task['relevant_files']}\n\n"
              f"STDOUT:\n{stdout[-2500:]}\n\nSTDERR:\n{stderr[-2500:]}")
    resp, decisions = route_and_call(Tier.COMPLEX, system, prompt, response_json=True)
    text = resp.text if hasattr(resp, "text") else str(resp)
    try:
        diagnosis = _extract_json(text)
    except Exception:
        diagnosis = {"category": "unknown", "root_cause": text[:300],
                     "relevant_files": task["relevant_files"], "repair_plan": "retry with more context"}
    console_log("DIAGNOSER", f"{diagnosis.get('category')}: {diagnosis.get('root_cause', '')[:120]}")
    return diagnosis


def repair_agent_run(task: dict, diagnosis: dict, state: ProjectState) -> dict:
    """DIAGNOSIS/REPAIR AGENT (repair phase): applies a targeted fix via the
    same controlled file tools the coding agent uses, driven by the diagnosis
    and the real error text -- not a full project regeneration."""
    system = ("You are the REPAIR AGENT. Apply a TARGETED fix for the diagnosed root cause "
              "using write_file. Do not rewrite unrelated files. Reply with ONLY a JSON "
              "object when done: {summary, files_changed}.")
    task_ctx = dict(task)
    task_ctx["relevant_files"] = diagnosis.get("relevant_files") or task["relevant_files"]
    task_ctx["last_error"] = f"{diagnosis.get('root_cause')}\nRepair plan: {diagnosis.get('repair_plan')}"
    context = build_task_context(task_ctx, state.as_dict())
    text, transcript, decisions = run_agent_with_tools(
        system, context, ["read_file", "search_code", "write_file", "file_exists"],
        Tier.COMPLEX, max_turns=6)
    console_log("REPAIR", f"Applying targeted fix for task '{task['title']}'")
    try:
        return _extract_json(text)
    except Exception:
        return {"summary": text[:300], "files_changed": [c["args"].get("path") for c in transcript
                                                          if c["tool"] == "write_file"]}


def replanner_run(state: ProjectState, graph: "TaskGraph", trigger_reason: str) -> dict:
    """REPLANNER: evaluates progress and may split/merge/reprioritize tasks,
    skip optional work, or mark tasks blocked. Applied conservatively -- only
    called when a task exhausts its repair budget or overall progress stalls."""
    system = ("You are the REPLANNER. Given current task graph state and why replanning was "
              "triggered, propose adjustments. Reply with ONLY a JSON object: {analysis, "
              "actions: array of {type (one of 'skip','reprioritize','block','add_task'), "
              "task_id (if applicable), new_priority (if reprioritize), new_task (if add_task: "
              "{title, description, acceptance_criteria})}}.")
    prompt = (f"Trigger: {trigger_reason}\n\nTasks:\n"
              f"{json.dumps([{k: t[k] for k in ('id','title','status','attempt_count','priority')} for t in state.tasks], indent=2)}")
    resp, decisions = route_and_call(Tier.NORMAL, system, prompt, response_json=True)
    text = resp.text if hasattr(resp, "text") else str(resp)
    try:
        plan = _extract_json(text)
    except Exception:
        plan = {"analysis": text[:300], "actions": []}
    for action in plan.get("actions", []):
        t = action.get("type")
        tid = action.get("task_id")
        if t == "skip" and tid:
            graph.set_status(tid, TaskStatus.SKIPPED)
        elif t == "block" and tid:
            graph.set_status(tid, TaskStatus.BLOCKED)
        elif t == "reprioritize" and tid:
            for task in state.tasks:
                if task["id"] == tid:
                    task["priority"] = action.get("new_priority", task["priority"])
        elif t == "add_task":
            nt = action.get("new_task", {})
            graph.add(title=nt.get("title", "Additional task"), description=nt.get("description", ""),
                      priority=1, acceptance_criteria=nt.get("acceptance_criteria", ""))
    console_log("REPLANNER", f"{len(plan.get('actions', []))} adjustments applied ({trigger_reason})")
    return plan


## Autonomous development engine

The closed loop that ties every component above together and enforces the resource limits from the Model Usage Monitor.


In [17]:
def _stop_reasons():
    reasons = MONITOR.limits_status()
    if len(STATE.completed_tasks) + len(STATE.failed_tasks) >= CONFIG["MAX_TASK_ITERATIONS"]:
        reasons.append("MAX_TASK_ITERATIONS reached")
    return reasons


def run_autonomous_engine(project_goal: str) -> ProjectState:
    """The closed loop: GOAL -> UNDERSTAND -> PLAN -> SELECT MODEL -> ACT ->
    EXECUTE -> OBSERVE -> DIAGNOSE -> REPAIR -> RETEST -> REPLAN -> VERIFY.
    Stops only on: objective satisfied, all tasks terminal, or a resource
    limit -- never runs unbounded."""
    global STATE, GRAPH
    STATE = ProjectState(project_goal)
    GRAPH = TaskGraph(STATE)

    console_log("MANAGER", f"Objective: {project_goal}")
    try:
        analysis = analyzer_run(PROJECT_MANIFEST)
        console_log("ANALYZER", analysis.get("summary", "")[:200])

        plan = manager_create_plan(project_goal, PROJECT_MANIFEST, analysis)
        materialize_plan(GRAPH, plan)

        while not GRAPH.all_terminal():
            stop = _stop_reasons()
            if stop:
                console_log("MANAGER", f"Stopping safely: {'; '.join(stop)}")
                STATE.status = "STOPPED_RESOURCE_LIMIT"
                break

            task = GRAPH.next_ready_task()
            if task is None:
                break
            GRAPH.set_status(task["id"], TaskStatus.RUNNING)
            STATE.execution_history.append({"t": elapsed(), "event": "task_started", "task": task["title"]})
            console_log("MANAGER", f"Selected task: {task['title']} (complexity={task['estimated_complexity']})")

            tier = {"complex": Tier.COMPLEX, "normal": Tier.NORMAL, "light": Tier.LIGHT}[task["estimated_complexity"]]
            console_log("ROUTER", f"Routing '{task['title']}' as {tier.value} complexity")

            create_checkpoint(label=f"before_{task['id']}")
            coding_result = coding_agent_run(task, STATE)
            task["attempt_count"] += 1

            test_result = testing_agent_run("tests")
            success = bool(test_result.get("success")) or (not test_result.get("ran") and
                       syntax_check(task["relevant_files"]).get("ok", False))

            repair_count = 0
            last_output_signature = None
            while not success and repair_count < CONFIG["MAX_REPAIR_ATTEMPTS"]:
                if _stop_reasons():
                    break
                signature = (test_result.get("failed"), test_result.get("stdout", "")[-200:])
                if signature == last_output_signature:
                    console_log("DIAGNOSER", "Repeated identical failure detected; escalating to replanner")
                    break
                last_output_signature = signature

                diagnosis = diagnose_failure(task, test_result)
                repair_agent_run(task, diagnosis, STATE)
                MONITOR.repair_attempts += 1
                repair_count += 1
                task["previous_attempt_summary"] = diagnosis.get("root_cause")

                test_result = testing_agent_run("tests")
                success = bool(test_result.get("success"))
                console_log("TESTER", "Re-running affected tests" if not success else "All tests passed")

            if success:
                GRAPH.set_status(task["id"], TaskStatus.COMPLETED, result=coding_result)
                STATE.completed_tasks.append(task["id"])
                console_log("MANAGER", f"Task completed: {task['title']}")
            else:
                GRAPH.set_status(task["id"], TaskStatus.FAILED,
                                  error=test_result.get("stderr", "")[-500:])
                STATE.failed_tasks.append(task["id"])
                console_log("MANAGER", f"Task failed after {repair_count} repair attempt(s): {task['title']}")
                replanner_run(STATE, GRAPH, trigger_reason=f"repeated failure on '{task['title']}'")

            STATE.model_usage = MONITOR.summary()
            STATE.save(WORKSPACE_ROOT / "project_state.json")

        if STATE.status == "IN_PROGRESS":
            STATE.status = "TASKS_EXHAUSTED"

    except ResourceLimitReached as e:
        console_log("MANAGER", f"Stopping safely: {e}")
        STATE.status = "STOPPED_RESOURCE_LIMIT"
    except Exception as e:
        console_log("MANAGER", f"Unrecoverable error: {e}")
        STATE.errors.append({"error": str(e), "trace": traceback.format_exc()[-2000:]})
        STATE.status = "FAILED"

    STATE.model_usage = MONITOR.summary()
    STATE.save(WORKSPACE_ROOT / "project_state.json")
    return STATE


## End-to-end demonstration

Runs the full loop on the built-in Expense Tracker. Expect at least one real autonomous correction: a test fails, the agent diagnoses the actual output, repairs the relevant file, and re-tests.


In [18]:
DEMO_OBJECTIVE = ("Improve this application, fix existing issues, add robust validation and "
                   "testing, improve error handling, and make the project production-ready "
                   "within the available scope.")

# This single call drives the entire closed loop described above: it will
# discover the validation, rounding, and truncation defects in the demo
# Expense Tracker on its own (they are not named here), attempt fixes,
# observe real test output, repair on failure, and re-test -- exactly the
# autonomous correction described in the notebook's design goals.
FINAL_STATE = run_autonomous_engine(DEMO_OBJECTIVE)


[MANAGER] Objective: Improve this application, fix existing issues, add robust validation and testing, improve error handling, and make the project production-ready within the available scope.
[ROUTER] gemini-2.5-flash failed (model_unavailable); selecting next model


[ANALYZER] Used 4 tool calls; producing findings
[ANALYZER] The project is a simple expense tracking application with a core logic layer, validation, and an in-memory database. I identified several logical defects regarding data accuracy and validation.
[PLANNER] 5 tasks created
[MANAGER] Selected task: Fix Precision Loss in Expense Calculations (complexity=light)
[ROUTER] Routing 'Fix Precision Loss in Expense Calculations' as light complexity
[SAFETY] Checkpoint created: 0_before_0618fb66
[ROUTER] gemini-3.1-flash-lite failed (transient); selecting next model
[ROUTER] gemini-3.1-flash-lite failed (transient); selecting next model
[CODER] Modified expenses.py
[CODER] Task 'Fix Precision Loss in Expense Calculations': 2 tool calls
[TESTER] 3 passed, 0 failed (exit=0)
[MANAGER] Task completed: Fix Precision Loss in Expense Calculations
[MANAGER] Selected task: Enhance Input Validation Logic (complexity=light)
[ROUTER] Routing 'Enhance Input Validation Logic' as light complexity
[SAFETY]

## Final validation, report, and packaging

Re-runs syntax/import/test checks before declaring a status, then writes the report, state, execution log, and final ZIP.


In [19]:
def final_validation(state: ProjectState) -> dict:
    """Actually re-runs syntax checks, imports, and the full test suite before
    any status is declared -- nothing here is asserted without evidence."""
    console_log("MANAGER", "Running final validation")
    all_py = [f for f in PROJECT_MANIFEST["source_files"] + PROJECT_MANIFEST["test_files"]]
    syntax = syntax_check(all_py)

    import_ok, import_errors = True, {}
    entry = PROJECT_MANIFEST.get("entry_points", [])
    if entry:
        r = run_python(["-c", f"import ast; ast.parse(open('{entry[0]}').read())"])
        import_ok = r["exit_code"] == 0
        if not import_ok:
            import_errors[entry[0]] = r["stderr"]

    test_result = testing_agent_run("tests")

    health_check = None
    if entry:
        health_check = run_python([entry[0]])

    passed_all = syntax["ok"] and import_ok and (test_result.get("success") if test_result.get("ran") else True)
    if not passed_all:
        final_status = "FAILED" if state.failed_tasks and not state.completed_tasks else "PARTIALLY VERIFIED"
    else:
        final_status = "VERIFIED" if not state.failed_tasks else "PARTIALLY VERIFIED"

    result = {
        "syntax_check": syntax, "import_check": {"ok": import_ok, "errors": import_errors},
        "test_result": test_result,
        "health_check": health_check,
        "final_status": final_status,
    }
    console_log("MANAGER", f"Final status: {final_status}")
    return result

VALIDATION_RESULT = final_validation(STATE)
STATE.status = VALIDATION_RESULT["final_status"]
STATE.save(WORKSPACE_ROOT / "project_state.json")


[MANAGER] Running final validation
[TESTER] 8 passed, 0 failed (exit=0)
[MANAGER] Final status: VERIFIED


In [20]:
def generate_report(state: ProjectState, validation: dict) -> str:
    tasks_by_status = {}
    for t in state.tasks:
        tasks_by_status.setdefault(t["status"], []).append(t["title"])
    usage = MONITOR.summary()

    lines = [
        "# Development Report", "",
        f"**Objective:** {state.project_goal}", "",
        f"**Final status:** {validation['final_status']}", "",
        "## Project overview",
        f"- Root: `{state.project_root}`",
        f"- Files: {state.project_manifest['total_files']} total, "
        f"{len(state.project_manifest['source_files'])} source, "
        f"{len(state.project_manifest['test_files'])} test",
        "",
        "## Task plan and outcomes",
    ]
    for status, titles in tasks_by_status.items():
        lines.append(f"- **{status}**: {', '.join(titles) if titles else '(none)'}")
    lines += ["", "## File changes",
              f"- Created: {FILE_LOG.created_files or '(none)'}",
              f"- Modified: {FILE_LOG.modified_files or '(none)'}",
              f"- Deleted: {FILE_LOG.deleted_files or '(none)'}",
              "", "## Tests executed"]
    for tr in state.test_results:
        lines.append(f"- scope={tr.get('scope')} passed={tr.get('passed')} "
                      f"failed={tr.get('failed')} exit={tr.get('exit_code')}")
    lines += ["", "## Repairs",
              f"- Total repair attempts: {usage['repair_attempts']}",
              "", "## Model usage",
              f"- Total model calls: {usage['total_calls']}",
              f"- Calls by model: {usage['calls_by_model']}",
              f"- Model switches: {usage['model_switches'] or '(none)'}",
              f"- Fallback calls: {usage['fallback_calls']}",
              f"- Input/output tokens: {usage['input_tokens']} / {usage['output_tokens']}",
              "", "## Resource-limit events",
              f"- {MONITOR.limits_status() or '(none triggered)'}",
              "", "## Final validation",
              f"- Syntax check ok: {validation['syntax_check']['ok']}",
              f"- Import check ok: {validation['import_check']['ok']}",
              f"- Test result: {validation['test_result']}",
              "", "## Remaining limitations",
              "- MVP execution/testing support is Python-only.",
              "- Local Qwen fallback requires a pre-downloaded GGUF (QWEN_GGUF_PATH); "
              "not auto-downloaded.",
              f"- {len(state.failed_tasks)} task(s) ended FAILED or SKIPPED after repair/replan.",
              ]
    report_text = "\n".join(lines)
    (WORKSPACE_ROOT / "development_report.md").write_text(report_text)
    return report_text


def package_outputs(state: ProjectState) -> Path:
    (WORKSPACE_ROOT / "execution_log.json").write_text(json.dumps(EXECUTION_LOG, indent=2, default=str))
    state.save(WORKSPACE_ROOT / "project_state.json")
    zip_path = WORKSPACE_ROOT / "project-final.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in PROJECT_ROOT.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(PROJECT_ROOT))
    console_log("MANAGER", f"Packaged {zip_path.name}")
    return zip_path


REPORT_TEXT = generate_report(STATE, VALIDATION_RESULT)
ZIP_PATH = package_outputs(STATE)

print("\n" + "=" * 60)
print("FINAL DASHBOARD")
print("=" * 60)
usage = MONITOR.summary()
dashboard = {
    "Project": PROJECT_ROOT.name,
    "Objective": STATE.project_goal,
    "Final Status": STATE.status,
    "Tasks Completed": len(STATE.completed_tasks),
    "Tasks Failed": len(STATE.failed_tasks),
    "Tests Passed": VALIDATION_RESULT["test_result"].get("passed"),
    "Tests Failed": VALIDATION_RESULT["test_result"].get("failed"),
    "Repair Attempts": usage["repair_attempts"],
    "Model Calls": usage["total_calls"],
    "Model Switches": len(usage["model_switches"]),
    "Current Model": usage["current_model"],
    "Execution Time (s)": usage["session_seconds"],
    "Files Modified": len(FILE_LOG.modified_files) + len(FILE_LOG.created_files),
}
for k, v in dashboard.items():
    print(f"{k:20s}: {v}")

print("\nOutput files:")
for f in ["project-final.zip", "development_report.md", "project_state.json", "execution_log.json"]:
    print(" -", WORKSPACE_ROOT / f)


[MANAGER] Packaged project-final.zip

FINAL DASHBOARD
Project             : project
Objective           : Improve this application, fix existing issues, add robust validation and testing, improve error handling, and make the project production-ready within the available scope.
Final Status        : VERIFIED
Tasks Completed     : 5
Tasks Failed        : 0
Tests Passed        : 8
Tests Failed        : 0
Repair Attempts     : 0
Model Calls         : 40
Model Switches      : 3
Current Model       : gemini-3.1-flash-lite
Execution Time (s)  : 418.5
Files Modified      : 9

Output files:
 - /content/autodev_workspace/project-final.zip
 - /content/autodev_workspace/development_report.md
 - /content/autodev_workspace/project_state.json
 - /content/autodev_workspace/execution_log.json
